# Rețele neuronale de la zero

Scriem de la zero, în Python, un motor de diferențiere automată, desenăm grafuri computaționale cu Graphviz și antrenăm rețele neuronale, până la recunoașterea cifrelor MNIST.

Notebookul rulează codul laboratorului [Rețele neuronale de la zero](https://inteligentartificiala.ro/laboratoare/retele-neuronale-de-la-zero/prezentare/), de pe [InteligentArtificiala.ro](https://inteligentartificiala.ro), unde este explicat fiecare fișier. Laboratorul nu are nevoie de o placă grafică: antrenarea pe MNIST durează câteva secunde pe procesor. Celulele se rulează în ordine, de sus în jos.

## Codul laboratorului

Fișierele se descarcă de pe site, ca în varianta cu `curl` de pe [pagina de introducere](https://inteligentartificiala.ro/laboratoare/retele-neuronale-de-la-zero/prezentare/#pregătirea-mediului-de-lucru). Colab are deja NumPy, Matplotlib și PyTorch. Diagramele au nevoie și de programul `dot`, din Graphviz, pe care ultima comandă îl instalează dacă lipsește.

In [ ]:
%%bash
mkdir -p retele-de-la-zero && cd retele-de-la-zero
for f in engine.py graph.py nn.py diagrams.py moons.py mnist_data.py mnist_numpy.py mnist_torch.py requirements.txt; do
  curl -fsSO "https://inteligentartificiala.ro/laboratoare/retele-neuronale-de-la-zero/cod/$f"
done
ls

In [ ]:
%cd retele-de-la-zero
%pip install -q -r requirements.txt
!which dot || (apt-get -qq update && apt-get -qq install -y graphviz)

## [Un motor de diferențiere automată](https://inteligentartificiala.ro/laboratoare/retele-neuronale-de-la-zero/motorul-de-diferentiere/)

Clasa `Value` din `engine.py` ține un număr, gradientul lui și operația din care a rezultat. Metoda `backward()` calculează gradienții tuturor valorilor din care a fost calculat rezultatul.

In [ ]:
from engine import Value

a = Value(2.0, label="a")
b = Value(-3.0, label="b")
c = Value(7.0, label="c")
f = Value(-2.0, label="f")
e = a * b
e.label = "e"
d = e + c
d.label = "d"
L = d * f
L.label = "L"

L.backward()
for v in [a, b, c, f]:
    print(f"dL/d{v.label} = {v.grad}")

## [Grafuri computaționale cu Graphviz](https://inteligentartificiala.ro/laboratoare/retele-neuronale-de-la-zero/grafuri-computationale/)

`draw_graph` din `graph.py` desenează graful unei valori. În Colab, diagrama apare direct sub celulă. Prima diagramă arată expresia înainte de retropropagare, cu toți gradienții 0, iar a doua, după `backward()`.

In [ ]:
from diagrams import expression
from graph import draw_graph

L = expression()
draw_graph(L)

In [ ]:
L.backward()
draw_graph(L)

Același lucru pentru un neuron cu activare tanh și pentru o rețea mică cu ReLU, cu pierderea pe un exemplu. A doua diagramă este lată: se poate derula orizontal.

In [ ]:
from diagrams import neuron

o = neuron()
o.backward()
draw_graph(o)

In [ ]:
from diagrams import tiny_mlp

loss = tiny_mlp()
loss.backward()
draw_graph(loss)

Scriptul `diagrams.py` salvează toate aceste diagrame ca fișiere SVG, în directorul `imagini/`.

In [ ]:
!python diagrams.py

## [O rețea neuronală mică](https://inteligentartificiala.ro/laboratoare/retele-neuronale-de-la-zero/o-retea-neuronala-mica/)

Un MLP construit din obiecte `Value`, cu 105 parametri, învață să separe două semilune de puncte. Antrenarea are 200 de pași și durează câteva zeci de secunde.

In [ ]:
!python moons.py

In [ ]:
from IPython.display import SVG

SVG("imagini/frontiera.svg")

## [Cifrele MNIST cu NumPy și PyTorch](https://inteligentartificiala.ro/laboratoare/retele-neuronale-de-la-zero/cifrele-mnist/)

Rețeaua 784 → 128 → 10, scrisă cu NumPy, cu retropropagarea scrisă de mână, se antrenează 10 epoci pe cele 60.000 de imagini de antrenare. Scriptul descarcă la prima rulare setul de date, de 11 MB.

In [ ]:
!python mnist_numpy.py

Primele 12 cifre din setul de testare, cu cifra prezisă deasupra, apoi primele 12 cifre clasificate greșit, cu cifra corectă între paranteze.

In [ ]:
from IPython.display import Image, display

display(Image("imagini/predictii.png"))
display(Image("imagini/greseli.png"))

Aceeași rețea în PyTorch. Scriptul compară întâi gradienții calculați de mână cu cei calculați de PyTorch, apoi antrenează rețeaua.

In [ ]:
!python mnist_torch.py